# VERITAS OW v4.1 (140k Full) — Proposed Decoupled Multi-Stream & Active Segmentation Model

Full 140,000-image OpenForensics Train run of VERITAS v4.1 (pruned dead Freq/MaskContrast, retained 9-ch color SRM + SpatialForensicEncoderV4 + Multi-Scale ActiveSegmentationHeadV4 + DecoupledForensicFusionV41 with bounded dual-expert logit fusion), evaluated on all 10,905 Test images across 5 robustness conditions.

- **Accelerator:** `GPU T4 x2` DDP
- **Data:** `ahmedgamal/openforensics-wild-dataset` + `clarenceshanley/veritas-ow-00-generate-masks`
- **Clean output:** Unpacks masks strictly to `/tmp/ow_masks_cache` so `/kaggle/working/` stays clean (~275 MB).


In [ ]:
import json
from pathlib import Path
import torch

WORKING = Path("/kaggle/working/ow_v41_140k_veritas_v4_1") if Path("/kaggle/working").exists() else Path.cwd() / "ow_v41_140k_veritas_v4_1"
WORKING.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "variant": 'veritas_v4_1',
    "working_dir": str(WORKING),
    "seed": 42,
    "image_size": 600,
    "train_per_class": None,   # FULL 140,000 OpenForensics Train images
    "val_per_class": 4000,     # 8,000 stratified Val images (fast epoch validation)
    "test_per_class": None,    # ALL 10,905 OpenForensics Test images
    "warmup_epochs": 1,        # Stage 1: frozen EfficientNet-B7, train heads/forensic/seg
    "finetune_epochs": 2,      # Stage 2: full end-to-end fine-tuning
    "batch_size_per_gpu": 4,   # 4 x 2 GPUs x 2 grad_accum = 16 effective batch
    "eval_batch_size_per_gpu": 8,
    "target_effective_batch": 16,
    "num_workers": 2,
    "use_amp": True,
    "use_forensic_aug": True,  # Identical forensic pipeline for both models
    "warmup_lr": 3e-4,
    "finetune_lr": 1.5e-4,
    "backbone_lr_multiplier": 0.1,
    "weight_decay": 1e-4,
    "lambda_cls": 1.0,
    "lambda_rgb": 0.50,
    "lambda_aux": 0.50,
    "lambda_seg": 0.50,
    "dice_weight": 1.0,
    "robustness_conditions": ["jpeg_q50", "jpeg_q30", "blur_sigma2", "resize_0.5x"],
    "robustness_time_budget_sec": 37800,  # 10.5 hrs safety guard under Kaggle 12h limit
}

cfg_path = WORKING / "config.json"
cfg_path.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"Variant={CONFIG['variant']} | GPUs={n_gpus}", [torch.cuda.get_device_name(i) for i in range(n_gpus)])
print("Saved config to:", cfg_path)


In [ ]:
# Write standalone DDP-safe full-140k training script
SCRIPT_CODE = 'from __future__ import annotations\nimport argparse\nimport gc\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport shutil\nimport time\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.distributed as dist\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import DataLoader, Dataset\nfrom torch.utils.data.distributed import DistributedSampler\nfrom torchvision.models import EfficientNet_B7_Weights, efficientnet_b7\n\n# === Embedded VERITAS v4 / v4.1 architecture ===\n"""VERITAS OW v4 & v4.1 architecture: zero-sync spatial forensic streams,\nmulti-scale deep-supervised segmentation, and decoupled dual-expert forensic fusion.\n\nKey architectural evolution:\n- OW v4 (`variant="veritas_v4"`):\n  1. Zero GPU->CPU syncs in ForensicStreamsV4 (vectorized eroded color SRM +\n     content-mean-filled batched 2D rfft2/irfft2 directional band-pass).\n  2. Spatial multi-scale SRM & frequency encoders (38x38 and 19x19).\n  3. Multi-scale ActiveSegmentationHeadV4 tapping EfficientNet-B7 stages 2, 5, 7\n     plus spatial forensic maps (38x38).\n  4. MaskGuidedContrastiveFusionV4 with 4-stream softmax gating + aux forensic head.\n\n- OW v4.1 (`variant="veritas_v4_1"` — empirically refined from 10,905-image v4\n  prototype diagnostics):\n  1. Prunes dead Freq FFT stream (learned gate 0.007) and heavy MaskContrast MLP\n     (learned gate 0.011), cutting ~40-50% of extra latency and ~1.5 hrs of 140k\n     training time while keeping the winning 9-ch color SRM + SpatialForensicEncoderV4\n     and ActiveSegmentationHeadV4 (Fake Dice ~0.89, mIoU ~0.85).\n  2. Fixes RGB-to-mask co-adaptation collapse (r(v4, Aux) = 0.9918 in v4) via\n     DecoupledForensicFusionV41:\n       - Pure Global RGB Expert (`rgb_logits`): valid-mask GAP over B7 stage 8\n         with ZERO segmentation mask input, preserving full B7 Fake recall.\n       - Pure Forensic SRM + Active Mask Expert (`aux_forensic_logits`): pools\n         spatial SRM over [global, fg, bg, |fg - bg|] + mask summary stats with\n         ZERO direct RGB features, preserving v4 Aux\'s strong Real FP rejection.\n       - Bounded Dual-Expert Logit Fusion (`cls_logits`): learns alpha strictly\n         within [0.35, 0.65] (initialized at 0.50) plus a small bounded (+/-0.25)\n         cross-modal residual, guaranteeing genuine RGB + Forensic complementarity.\n"""\n\nimport math\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torchvision.models import EfficientNet_B7_Weights, efficientnet_b7\n\nimport numpy as np\n\nIMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)\nIMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)\n\n\nclass ClassificationHead(nn.Module):\n    def __init__(self, in_ch: int = 2560, dropout: float = 0.2):\n        super().__init__()\n        self.pool = nn.AdaptiveAvgPool2d(1)\n        self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_ch, 1))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.fc(self.pool(x).flatten(1))\n\n\nIMPLEMENTATION_VERSION = "ow_v4_1"\n\n\ndef _srm_kernels_v4() -> torch.Tensor:\n    """Pure-PyTorch 3x1x5x5 SRM high-pass kernels (NumPy-version independent)."""\n    raw = torch.tensor(\n        [\n            [\n                [0, 0, 0, 0, 0],\n                [0, -1, 2, -1, 0],\n                [0, 2, -4, 2, 0],\n                [0, -1, 2, -1, 0],\n                [0, 0, 0, 0, 0],\n            ],\n            [\n                [-1, 2, -2, 2, -1],\n                [2, -6, 8, -6, 2],\n                [-2, 8, -12, 8, -2],\n                [2, -6, 8, -6, 2],\n                [-1, 2, -2, 2, -1],\n            ],\n            [\n                [0, 0, 0, 0, 0],\n                [0, 0, 0, 0, 0],\n                [0, 1, -2, 1, 0],\n                [0, 0, 0, 0, 0],\n                [0, 0, 0, 0, 0],\n            ],\n        ],\n        dtype=torch.float32,\n    )\n    norms = raw.abs().sum(dim=(-2, -1), keepdim=True).clamp_min(1.0)\n    return (raw / norms).unsqueeze(1)\n\n\ndef _gn(channels: int) -> nn.GroupNorm:\n    return nn.GroupNorm(math.gcd(32, channels), channels)\n\n\nclass SeparableConvBlock(nn.Sequential):\n    def __init__(self, in_ch: int, out_ch: int, stride: int = 1, dilation: int = 1):\n        super().__init__(\n            nn.Conv2d(\n                in_ch,\n                in_ch,\n                kernel_size=3,\n                stride=stride,\n                padding=dilation,\n                dilation=dilation,\n                groups=in_ch,\n                bias=False,\n            ),\n            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),\n            _gn(out_ch),\n            nn.SiLU(inplace=True),\n        )\n\n\nclass ResidualForensicBlock(nn.Module):\n    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):\n        super().__init__()\n        self.conv1 = SeparableConvBlock(in_ch, out_ch, stride=stride)\n        self.conv2 = nn.Sequential(\n            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, groups=out_ch, bias=False),\n            nn.Conv2d(out_ch, out_ch, kernel_size=1, bias=False),\n            _gn(out_ch),\n        )\n        self.skip = (\n            nn.Identity()\n            if (stride == 1 and in_ch == out_ch)\n            else nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), _gn(out_ch))\n        )\n        self.act = nn.SiLU(inplace=True)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.act(self.conv2(self.conv1(x)) + self.skip(x))\n\n\nclass SRMStreamV41(nn.Module):\n    """Vectorized, zero-sync 9-channel color SRM residual extractor (v4.1).\n\n    Applies 3 fixed 5x5 SRM high-pass filters per RGB channel with tanh\n    bounding and 5x5 eroded valid_mask gating so letterbox borders never\n    produce spurious high-pass spikes.\n    """\n\n    def __init__(self):\n        super().__init__()\n        self.register_buffer("srm_weight", _srm_kernels_v4())\n\n    @staticmethod\n    def _erode_valid_mask(valid_mask: torch.Tensor) -> torch.Tensor:\n        return -F.max_pool2d(-valid_mask, kernel_size=5, stride=1, padding=2)\n\n    def forward(self, normalized_rgb: torch.Tensor, valid_mask: torch.Tensor | None = None) -> torch.Tensor:\n        with torch.autocast(device_type=normalized_rgb.device.type, enabled=False):\n            x = normalized_rgb.float()\n            mean = IMAGENET_MEAN.to(device=x.device, dtype=torch.float32)\n            std = IMAGENET_STD.to(device=x.device, dtype=torch.float32)\n            rgb = (x * std + mean).clamp(0.0, 1.0)\n\n            if valid_mask is None:\n                eroded = torch.ones((rgb.shape[0], 1, rgb.shape[-2], rgb.shape[-1]), device=rgb.device, dtype=torch.float32)\n                rgb_filled = rgb\n            else:\n                vm = valid_mask.to(device=rgb.device, dtype=torch.float32)\n                eroded = self._erode_valid_mask(vm)\n                denom = vm.sum(dim=(-2, -1), keepdim=True).clamp_min(1.0)\n                content_mean = (rgb * vm).sum(dim=(-2, -1), keepdim=True) / denom\n                rgb_filled = rgb * vm + content_mean * (1.0 - vm)\n\n            srm_w = self.srm_weight.float().repeat(3, 1, 1, 1)\n            srm = F.conv2d(rgb_filled, srm_w, padding=2, groups=3).tanh() * eroded\n        return srm\n\n\nclass ForensicStreamsV4(nn.Module):\n    """Vectorized, zero-sync color SRM + spatial directional FFT band-pass (v4)."""\n\n    def __init__(self):\n        super().__init__()\n        self.register_buffer("srm_weight", _srm_kernels_v4())\n        self._band_cache: dict[tuple[int, int, str, int | None], torch.Tensor] = {}\n\n    def _get_freq_masks(self, h: int, w: int, device: torch.device) -> torch.Tensor:\n        key = (h, w, device.type, device.index)\n        cached = self._band_cache.get(key)\n        if cached is not None:\n            return cached\n        fy = torch.fft.fftfreq(h, device=device)[:, None]\n        fx = torch.fft.rfftfreq(w, device=device)[None, :]\n        radius = torch.sqrt(fy.square() + fx.square())\n        abs_fy = fy.abs()\n        abs_fx = fx.abs()\n\n        low_cut = torch.sigmoid((radius - 0.08) * 40.0)\n        mid_hi = torch.sigmoid((radius - 0.25) * 40.0)\n        mid_band = low_cut * (1.0 - mid_hi)\n\n        hv_ratio = torch.abs(abs_fx - abs_fy) / (abs_fx + abs_fy + 1e-6)\n        high_hv = mid_hi * hv_ratio\n        high_diag = mid_hi * (1.0 - hv_ratio)\n\n        masks = torch.stack([mid_band, high_hv, high_diag], dim=0)\n        if len(self._band_cache) > 8:\n            self._band_cache.clear()\n        self._band_cache[key] = masks\n        return masks\n\n    @staticmethod\n    def _erode_valid_mask(valid_mask: torch.Tensor) -> torch.Tensor:\n        return -F.max_pool2d(-valid_mask, kernel_size=5, stride=1, padding=2)\n\n    def forward(\n        self, normalized_rgb: torch.Tensor, valid_mask: torch.Tensor | None = None\n    ) -> tuple[torch.Tensor, torch.Tensor]:\n        with torch.autocast(device_type=normalized_rgb.device.type, enabled=False):\n            x = normalized_rgb.float()\n            mean = IMAGENET_MEAN.to(device=x.device, dtype=torch.float32)\n            std = IMAGENET_STD.to(device=x.device, dtype=torch.float32)\n            rgb = (x * std + mean).clamp(0.0, 1.0)\n\n            if valid_mask is None:\n                vm = torch.ones((rgb.shape[0], 1, rgb.shape[-2], rgb.shape[-1]), device=rgb.device, dtype=torch.float32)\n                eroded = vm\n                rgb_filled = rgb\n            else:\n                vm = valid_mask.to(device=rgb.device, dtype=torch.float32)\n                eroded = self._erode_valid_mask(vm)\n                denom = vm.sum(dim=(-2, -1), keepdim=True).clamp_min(1.0)\n                content_mean = (rgb * vm).sum(dim=(-2, -1), keepdim=True) / denom\n                rgb_filled = rgb * vm + content_mean * (1.0 - vm)\n\n            srm_w = self.srm_weight.float().repeat(3, 1, 1, 1)\n            srm = F.conv2d(rgb_filled, srm_w, padding=2, groups=3).tanh() * eroded\n\n            h, w = rgb_filled.shape[-2:]\n            spec = torch.fft.rfft2(rgb_filled, norm="ortho")\n            masks = self._get_freq_masks(h, w, rgb_filled.device)\n            filtered = (spec.unsqueeze(2) * masks.unsqueeze(0).unsqueeze(0)).flatten(1, 2)\n            spatial_freq = torch.fft.irfft2(filtered, s=(h, w), norm="ortho")\n            freq = torch.tanh(spatial_freq * 8.0) * eroded\n\n        return srm, freq\n\n\nclass SpatialForensicEncoderV4(nn.Module):\n    """Multi-scale spatial encoder aligned with EfficientNet-B7 stages 5 & 7/8.\n\n    Produces:\n      - `mid` at ~H/16 (38x38 for 600px, 128 ch) -> feeds ASPP segmentation head\n      - `top` at ~H/32 (19x19 for 600px, 256 ch) -> feeds forensic classification\n    """\n\n    def __init__(self, in_ch: int = 9, mid_ch: int = 128, top_ch: int = 256):\n        super().__init__()\n        self.stem = nn.Sequential(\n            nn.Conv2d(in_ch, 32, kernel_size=3, stride=2, padding=1, bias=False),\n            _gn(32),\n            nn.SiLU(inplace=True),\n        )\n        self.stage1 = ResidualForensicBlock(32, 64, stride=2)   # /4  (150)\n        self.stage2 = ResidualForensicBlock(64, 96, stride=2)   # /8  (75)\n        self.stage3 = nn.Sequential(\n            ResidualForensicBlock(96, mid_ch, stride=2),        # /16 (38)\n            ResidualForensicBlock(mid_ch, mid_ch, stride=1),\n        )\n        self.stage4 = nn.Sequential(\n            ResidualForensicBlock(mid_ch, top_ch, stride=2),    # /32 (19)\n            ResidualForensicBlock(top_ch, top_ch, stride=1),\n        )\n\n    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n        x = self.stem(x)\n        x = self.stage1(x)\n        x = self.stage2(x)\n        mid = self.stage3(x)\n        top = self.stage4(mid)\n        return mid, top\n\n\nclass MultiScaleSeparableASPP(nn.Module):\n    def __init__(self, in_ch: int, out_ch: int = 256, rates: tuple[int, ...] = (3, 6, 12)):\n        super().__init__()\n        self.branches = nn.ModuleList(\n            [nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), _gn(out_ch), nn.SiLU(inplace=True))]\n        )\n        self.branches.extend(SeparableConvBlock(in_ch, out_ch, dilation=r) for r in rates)\n        self.gap = nn.Sequential(\n            nn.AdaptiveAvgPool2d(1),\n            nn.Conv2d(in_ch, out_ch, 1, bias=False),\n            _gn(out_ch),\n            nn.SiLU(inplace=True),\n        )\n        self.project = nn.Sequential(\n            nn.Conv2d(out_ch * (len(rates) + 2), out_ch, 1, bias=False),\n            _gn(out_ch),\n            nn.SiLU(inplace=True),\n            nn.Dropout2d(0.1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        feats = [b(x) for b in self.branches]\n        feats.append(F.interpolate(self.gap(x), size=x.shape[-2:], mode="bilinear", align_corners=False))\n        return self.project(torch.cat(feats, dim=1))\n\n\nclass ActiveSegmentationHeadV4(nn.Module):\n    """Deeply supervised multi-scale segmentation decoder.\n\n    Fuses EfficientNet-B7 stage 7 (640 ch, 19x19) + stage 5 (224 ch, 38x38) +\n    spatial forensic maps (128 ch for v4.1 SRM-only, or 256 ch for v4 SRM+Freq)\n    before ASPP, then refines with stage 2 (48 ch, 150x150).\n    """\n\n    def __init__(\n        self,\n        high_ch: int = 640,\n        mid_ch: int = 224,\n        low_ch: int = 48,\n        forensic_mid_ch: int = 128,\n        use_forensic: bool = True,\n        num_forensic_streams: int = 2,\n    ):\n        super().__init__()\n        self.use_forensic = bool(use_forensic)\n        self.num_forensic_streams = int(num_forensic_streams)\n        self.high_proj = nn.Sequential(nn.Conv2d(high_ch, 192, 1, bias=False), _gn(192), nn.SiLU(inplace=True))\n        self.mid_proj = nn.Sequential(nn.Conv2d(mid_ch, 128, 1, bias=False), _gn(128), nn.SiLU(inplace=True))\n        fused_ch = 192 + 128\n        if self.use_forensic:\n            self.forensic_proj = nn.Sequential(\n                nn.Conv2d(forensic_mid_ch * self.num_forensic_streams, 128, 1, bias=False),\n                _gn(128),\n                nn.SiLU(inplace=True),\n            )\n            fused_ch += 128\n        self.aspp = MultiScaleSeparableASPP(fused_ch, out_ch=256, rates=(3, 6, 12))\n        self.low_proj = nn.Sequential(nn.Conv2d(low_ch, 48, 1, bias=False), _gn(48), nn.SiLU(inplace=True))\n        self.decoder = nn.Sequential(\n            SeparableConvBlock(256 + 48, 192),\n            SeparableConvBlock(192, 128),\n            nn.Conv2d(128, 1, kernel_size=1),\n        )\n\n    def forward(\n        self,\n        high: torch.Tensor,\n        mid: torch.Tensor,\n        low: torch.Tensor,\n        out_size: tuple[int, int],\n        srm_mid: torch.Tensor | None = None,\n        freq_mid: torch.Tensor | None = None,\n    ) -> torch.Tensor:\n        target_mid = mid.shape[-2:]\n        parts = [\n            F.interpolate(self.high_proj(high), size=target_mid, mode="bilinear", align_corners=False),\n            self.mid_proj(mid),\n        ]\n        if self.use_forensic and srm_mid is not None:\n            if srm_mid.shape[-2:] != target_mid:\n                srm_mid = F.interpolate(srm_mid, size=target_mid, mode="bilinear", align_corners=False)\n            if self.num_forensic_streams == 1:\n                parts.append(self.forensic_proj(srm_mid))\n            elif freq_mid is not None:\n                if freq_mid.shape[-2:] != target_mid:\n                    freq_mid = F.interpolate(freq_mid, size=target_mid, mode="bilinear", align_corners=False)\n                parts.append(self.forensic_proj(torch.cat([srm_mid, freq_mid], dim=1)))\n\n        aspp_out = self.aspp(torch.cat(parts, dim=1))\n        up = F.interpolate(aspp_out, size=low.shape[-2:], mode="bilinear", align_corners=False)\n        logits = self.decoder(torch.cat([up, self.low_proj(low)], dim=1))\n        return F.interpolate(logits, size=out_size, mode="bilinear", align_corners=False)\n\n\nclass MaskGuidedContrastiveFusionV4(nn.Module):\n    """Original v4 4-stream contrastive fusion (kept for backward compatibility)."""\n\n    def __init__(self, rgb_ch: int = 2560, forensic_ch: int = 256, embed_dim: int = 256, dropout: float = 0.25):\n        super().__init__()\n        self.rgb_reduce = nn.Sequential(\n            nn.Conv2d(rgb_ch, embed_dim, kernel_size=1, bias=False),\n            _gn(embed_dim),\n            nn.SiLU(inplace=True),\n        )\n        self.rgb_proj = self._stream_mlp(embed_dim * 4, embed_dim, dropout)\n        self.srm_proj = self._stream_mlp(forensic_ch * 4, embed_dim, dropout)\n        self.freq_proj = self._stream_mlp(forensic_ch * 4, embed_dim, dropout)\n        self.contrast_proj = self._stream_mlp(embed_dim + forensic_ch * 2 + 4, embed_dim, dropout)\n\n        self.gate_net = nn.Sequential(\n            nn.LayerNorm(embed_dim * 4),\n            nn.Linear(embed_dim * 4, 128),\n            nn.SiLU(inplace=True),\n            nn.Linear(128, 4),\n        )\n        self.classifier = nn.Sequential(\n            nn.LayerNorm(embed_dim * 4),\n            nn.Dropout(dropout),\n            nn.Linear(embed_dim * 4, embed_dim),\n            nn.SiLU(inplace=True),\n            nn.Dropout(dropout),\n            nn.Linear(embed_dim, 1),\n        )\n        self.aux_forensic_cls = nn.Sequential(\n            nn.LayerNorm(embed_dim * 2 + 4),\n            nn.Dropout(dropout),\n            nn.Linear(embed_dim * 2 + 4, 128),\n            nn.SiLU(inplace=True),\n            nn.Linear(128, 1),\n        )\n\n    @staticmethod\n    def _stream_mlp(in_dim: int, out_dim: int, dropout: float) -> nn.Sequential:\n        return nn.Sequential(\n            nn.LayerNorm(in_dim),\n            nn.Dropout(dropout),\n            nn.Linear(in_dim, out_dim),\n            nn.SiLU(inplace=True),\n            nn.LayerNorm(out_dim),\n        )\n\n    @staticmethod\n    def _masked_stats(\n        feat: torch.Tensor, mask_prob: torch.Tensor, valid: torch.Tensor\n    ) -> tuple[torch.Tensor, torch.Tensor]:\n        eps = 1e-5\n        w_glob = valid\n        w_fg = mask_prob * valid\n        w_bg = (1.0 - mask_prob) * valid\n\n        glob = (feat * w_glob).sum(dim=(-2, -1)) / w_glob.sum(dim=(-2, -1)).clamp_min(eps)\n        fg = (feat * w_fg).sum(dim=(-2, -1)) / w_fg.sum(dim=(-2, -1)).clamp_min(eps)\n        bg = (feat * w_bg).sum(dim=(-2, -1)) / w_bg.sum(dim=(-2, -1)).clamp_min(eps)\n        fg_frac = w_fg.sum(dim=(-2, -1)) / w_glob.sum(dim=(-2, -1)).clamp_min(eps)\n        bg_frac = w_bg.sum(dim=(-2, -1)) / w_glob.sum(dim=(-2, -1)).clamp_min(eps)\n        contrast_gate = (4.0 * fg_frac * bg_frac).clamp(0.0, 1.0)\n        diff = (fg - bg).abs() * contrast_gate\n        return torch.cat([glob, fg, bg, diff], dim=1), diff\n\n    def forward(\n        self,\n        rgb_top: torch.Tensor,\n        srm_top: torch.Tensor,\n        freq_top: torch.Tensor,\n        seg_prob: torch.Tensor,\n        valid_mask: torch.Tensor | None = None,\n    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        grid = rgb_top.shape[-2:]\n        rgb_feat = self.rgb_reduce(rgb_top)\n        if srm_top.shape[-2:] != grid:\n            srm_top = F.interpolate(srm_top, size=grid, mode="bilinear", align_corners=False)\n        if freq_top.shape[-2:] != grid:\n            freq_top = F.interpolate(freq_top, size=grid, mode="bilinear", align_corners=False)\n\n        m = F.interpolate(seg_prob.float(), size=grid, mode="bilinear", align_corners=False).to(rgb_feat.dtype)\n        if valid_mask is None:\n            v = torch.ones_like(m)\n        else:\n            v = F.interpolate(valid_mask.float(), size=grid, mode="area").clamp(0.0, 1.0).to(rgb_feat.dtype)\n            v = (v > 0.1).to(rgb_feat.dtype)\n\n        rgb_vec, rgb_diff = self._masked_stats(rgb_feat, m, v)\n        srm_vec, srm_diff = self._masked_stats(srm_top, m, v)\n        freq_vec, freq_diff = self._masked_stats(freq_top, m, v)\n\n        v_sum = v.sum(dim=(-2, -1)).clamp_min(1e-5)\n        m_mean = (m * v).sum(dim=(-2, -1)) / v_sum\n        m_max = (m * v).amax(dim=(-2, -1))\n        m_var = (((m - m_mean[:, :, None, None]) ** 2) * v).sum(dim=(-2, -1)) / v_sum\n        m_std = torch.sqrt(m_var.clamp_min(1e-6))\n        m_hi = ((m > 0.5).to(m.dtype) * v).sum(dim=(-2, -1)) / v_sum\n        mask_stats = torch.cat([m_mean, m_max, m_std, m_hi], dim=1)\n\n        t_rgb = self.rgb_proj(rgb_vec)\n        t_srm = self.srm_proj(srm_vec)\n        t_freq = self.freq_proj(freq_vec)\n        t_con = self.contrast_proj(torch.cat([rgb_diff, srm_diff, freq_diff, mask_stats], dim=1))\n\n        tokens = torch.cat([t_rgb, t_srm, t_freq, t_con], dim=1)\n        gates = torch.softmax(self.gate_net(tokens), dim=1)\n        gated_tokens = torch.cat(\n            [\n                t_rgb * (1.0 + gates[:, 0:1]),\n                t_srm * (1.0 + gates[:, 1:2]),\n                t_freq * (1.0 + gates[:, 2:3]),\n                t_con * (1.0 + gates[:, 3:4]),\n            ],\n            dim=1,\n        )\n        cls_logits = self.classifier(gated_tokens)\n        aux_forensic_logits = self.aux_forensic_cls(torch.cat([t_srm, t_freq, mask_stats], dim=1))\n        return cls_logits, aux_forensic_logits, gates\n\n\nclass DecoupledForensicFusionV41(nn.Module):\n    """Decoupled dual-expert logit fusion with bounded complementarity (v4.1).\n\n    Prevents RGB-to-mask co-adaptation collapse by keeping two strictly\n    separated experts:\n      1) Pure Global RGB Expert (`rgb_logits`): valid-mask-aware GAP over B7\n         stage 8 (2560-d) with ZERO segmentation mask input.\n      2) Pure Forensic SRM + Active Mask Expert (`aux_forensic_logits`):\n         mask-guided [global, fg, bg, |fg - bg|] pooling over spatial SRM\n         features (256-d) + 4 valid-area mask summary statistics, with ZERO\n         direct RGB features.\n      3) Bounded Logit Fusion (`cls_logits`):\n           cls_logits = alpha * rgb_logits + (1 - alpha) * aux_forensic_logits\n                        + res_scale * tanh(res_net([rgb, aux, |rgb-aux|, mask_stats]))\n         where alpha is strictly bounded to [0.35, 0.65] (init 0.50) and\n         res_scale is bounded to [-0.25, +0.25] (init ~0.025).\n    """\n\n    def __init__(self, rgb_ch: int = 2560, forensic_ch: int = 256, embed_dim: int = 256, dropout: float = 0.25):\n        super().__init__()\n        self.rgb_head = nn.Sequential(\n            nn.Dropout(0.2),\n            nn.Linear(rgb_ch, 1),\n        )\n        self.srm_proj = nn.Sequential(\n            nn.LayerNorm(forensic_ch * 4),\n            nn.Dropout(dropout),\n            nn.Linear(forensic_ch * 4, embed_dim),\n            nn.SiLU(inplace=True),\n            nn.LayerNorm(embed_dim),\n        )\n        self.forensic_head = nn.Sequential(\n            nn.LayerNorm(embed_dim + 4),\n            nn.Dropout(dropout),\n            nn.Linear(embed_dim + 4, 128),\n            nn.SiLU(inplace=True),\n            nn.Linear(128, 1),\n        )\n        # Learnable fusion weight bounded strictly to [0.35, 0.65], initialized at 0.50\n        self.raw_alpha = nn.Parameter(torch.zeros(1))\n        # Lightweight bounded cross-modal residual (max +/- 0.25 logit adjustment)\n        self.raw_res_scale = nn.Parameter(torch.full((1,), 0.1))\n        self.res_net = nn.Sequential(\n            nn.LayerNorm(7),\n            nn.Linear(7, 32),\n            nn.SiLU(inplace=True),\n            nn.Dropout(0.2),\n            nn.Linear(32, 1),\n        )\n\n    @staticmethod\n    def _masked_stats(\n        feat: torch.Tensor, mask_prob: torch.Tensor, valid: torch.Tensor\n    ) -> torch.Tensor:\n        eps = 1e-5\n        w_glob = valid\n        w_fg = mask_prob * valid\n        w_bg = (1.0 - mask_prob) * valid\n\n        glob = (feat * w_glob).sum(dim=(-2, -1)) / w_glob.sum(dim=(-2, -1)).clamp_min(eps)\n        fg = (feat * w_fg).sum(dim=(-2, -1)) / w_fg.sum(dim=(-2, -1)).clamp_min(eps)\n        bg = (feat * w_bg).sum(dim=(-2, -1)) / w_bg.sum(dim=(-2, -1)).clamp_min(eps)\n        fg_frac = w_fg.sum(dim=(-2, -1)) / w_glob.sum(dim=(-2, -1)).clamp_min(eps)\n        bg_frac = w_bg.sum(dim=(-2, -1)) / w_glob.sum(dim=(-2, -1)).clamp_min(eps)\n        contrast_gate = (4.0 * fg_frac * bg_frac).clamp(0.0, 1.0)\n        diff = (fg - bg).abs() * contrast_gate\n        return torch.cat([glob, fg, bg, diff], dim=1)\n\n    def forward(\n        self,\n        rgb_top: torch.Tensor,\n        srm_top: torch.Tensor,\n        seg_prob: torch.Tensor,\n        valid_mask: torch.Tensor | None = None,\n    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:\n        grid = rgb_top.shape[-2:]\n        if srm_top.shape[-2:] != grid:\n            srm_top = F.interpolate(srm_top, size=grid, mode="bilinear", align_corners=False)\n\n        m = F.interpolate(seg_prob.float(), size=grid, mode="bilinear", align_corners=False).to(rgb_top.dtype)\n        if valid_mask is None:\n            v = torch.ones_like(m)\n        else:\n            v = F.interpolate(valid_mask.float(), size=grid, mode="area").clamp(0.0, 1.0).to(rgb_top.dtype)\n            v = (v > 0.1).to(rgb_top.dtype)\n\n        v_sum = v.sum(dim=(-2, -1)).clamp_min(1e-5)\n\n        # Expert 1: Pure Global RGB (no segmentation mask gating!)\n        rgb_glob = (rgb_top * v).sum(dim=(-2, -1)) / v_sum\n        rgb_logits = self.rgb_head(rgb_glob)\n\n        # Expert 2: Pure SRM + Active Mask Forensics (no RGB features!)\n        srm_vec = self._masked_stats(srm_top, m, v)\n        m_mean = (m * v).sum(dim=(-2, -1)) / v_sum\n        m_max = (m * v).amax(dim=(-2, -1))\n        m_var = (((m - m_mean[:, :, None, None]) ** 2) * v).sum(dim=(-2, -1)) / v_sum\n        m_std = torch.sqrt(m_var.clamp_min(1e-6))\n        m_hi = ((m > 0.5).to(m.dtype) * v).sum(dim=(-2, -1)) / v_sum\n        mask_stats = torch.cat([m_mean, m_max, m_std, m_hi], dim=1)\n\n        t_srm = self.srm_proj(srm_vec)\n        aux_forensic_logits = self.forensic_head(torch.cat([t_srm, mask_stats], dim=1))\n\n        # Bounded Dual-Expert Logit Fusion + small cross-modal residual\n        alpha = 0.35 + 0.30 * torch.sigmoid(self.raw_alpha)\n        res_scale = 0.25 * torch.tanh(self.raw_res_scale)\n        res_in = torch.cat(\n            [\n                rgb_logits.detach(),\n                aux_forensic_logits.detach(),\n                (rgb_logits - aux_forensic_logits).abs().detach(),\n                mask_stats,\n            ],\n            dim=1,\n        )\n        cls_logits = (\n            alpha * rgb_logits\n            + (1.0 - alpha) * aux_forensic_logits\n            + res_scale * torch.tanh(self.res_net(res_in))\n        )\n        n = rgb_top.shape[0]\n        gates = torch.cat([alpha.expand(n, 1), (1.0 - alpha).expand(n, 1)], dim=1)\n        return cls_logits, rgb_logits, aux_forensic_logits, gates\n\n\nclass VERITASV4(nn.Module):\n    """Unified OW v4 / v4.1 model supporting `baseline`, `veritas_v4`, and `veritas_v4_1`."""\n\n    def __init__(self, variant: str = "veritas_v4_1", pretrained: bool = True, dropout: float = 0.25):\n        super().__init__()\n        if variant not in ("baseline", "veritas_v4", "veritas_v4_1"):\n            raise ValueError("variant must be \'baseline\', \'veritas_v4\', or \'veritas_v4_1\'")\n        self.variant = variant\n        self.multi_stream = variant in ("veritas_v4", "veritas_v4_1")\n        self.segmentation = variant in ("veritas_v4", "veritas_v4_1")\n\n        weights = EfficientNet_B7_Weights.IMAGENET1K_V1 if pretrained else None\n        self.encoder = efficientnet_b7(weights=weights).features\n\n        if variant == "baseline":\n            self.cls_head = ClassificationHead(2560)\n            self.streams = None\n            self.srm_encoder = None\n            self.freq_encoder = None\n            self.seg_head = None\n            self.fusion_head = None\n        elif variant == "veritas_v4":\n            self.cls_head = None\n            self.streams = ForensicStreamsV4()\n            self.srm_encoder = SpatialForensicEncoderV4(in_ch=9, mid_ch=128, top_ch=256)\n            self.freq_encoder = SpatialForensicEncoderV4(in_ch=9, mid_ch=128, top_ch=256)\n            self.seg_head = ActiveSegmentationHeadV4(\n                high_ch=640, mid_ch=224, low_ch=48, forensic_mid_ch=128, use_forensic=True, num_forensic_streams=2\n            )\n            self.fusion_head = MaskGuidedContrastiveFusionV4(\n                rgb_ch=2560, forensic_ch=256, embed_dim=256, dropout=dropout\n            )\n        else:  # veritas_v4_1\n            self.cls_head = None\n            self.streams = SRMStreamV41()\n            self.srm_encoder = SpatialForensicEncoderV4(in_ch=9, mid_ch=128, top_ch=256)\n            self.freq_encoder = None\n            self.seg_head = ActiveSegmentationHeadV4(\n                high_ch=640, mid_ch=224, low_ch=48, forensic_mid_ch=128, use_forensic=True, num_forensic_streams=1\n            )\n            self.fusion_head = DecoupledForensicFusionV41(\n                rgb_ch=2560, forensic_ch=256, embed_dim=256, dropout=dropout\n            )\n\n    def freeze_backbone(self, freeze: bool = True) -> None:\n        for p in self.encoder.parameters():\n            p.requires_grad = not freeze\n\n    def forward(self, x: torch.Tensor, valid_mask: torch.Tensor | None = None) -> dict[str, torch.Tensor]:\n        size = x.shape[-2:]\n        low = mid = high = None\n        feat = x\n        for i, stage in enumerate(self.encoder):\n            feat = stage(feat)\n            if i == 2:\n                low = feat\n            elif i == 5:\n                mid = feat\n            elif i == 7:\n                high = feat\n\n        if self.variant == "baseline":\n            logits = self.cls_head(feat)\n            return {\n                "classification_logits": logits,\n                "fake_probability": logits.sigmoid(),\n            }\n\n        if self.variant == "veritas_v4":\n            srm_map, freq_map = self.streams(x, valid_mask)\n            srm_mid, srm_top = self.srm_encoder(srm_map)\n            freq_mid, freq_top = self.freq_encoder(freq_map)\n\n            seg_logits = self.seg_head(high, mid, low, out_size=size, srm_mid=srm_mid, freq_mid=freq_mid)\n            seg_prob = seg_logits.sigmoid()\n\n            cls_logits, aux_forensic_logits, stream_gates = self.fusion_head(\n                feat, srm_top, freq_top, seg_prob, valid_mask=valid_mask\n            )\n            return {\n                "classification_logits": cls_logits,\n                "fake_probability": cls_logits.sigmoid(),\n                "aux_forensic_logits": aux_forensic_logits,\n                "segmentation_logits": seg_logits,\n                "segmentation_prob": seg_prob,\n                "stream_gates": stream_gates,\n            }\n\n        # veritas_v4_1\n        srm_map = self.streams(x, valid_mask)\n        srm_mid, srm_top = self.srm_encoder(srm_map)\n\n        seg_logits = self.seg_head(high, mid, low, out_size=size, srm_mid=srm_mid)\n        seg_prob = seg_logits.sigmoid()\n\n        cls_logits, rgb_logits, aux_forensic_logits, stream_gates = self.fusion_head(\n            feat, srm_top, seg_prob, valid_mask=valid_mask\n        )\n        return {\n            "classification_logits": cls_logits,\n            "fake_probability": cls_logits.sigmoid(),\n            "rgb_logits": rgb_logits,\n            "aux_forensic_logits": aux_forensic_logits,\n            "segmentation_logits": seg_logits,\n            "segmentation_prob": seg_prob,\n            "stream_gates": stream_gates,\n        }\n\n\nIMAGENET_MEAN_NP = np.array([0.485, 0.456, 0.406], dtype=np.float32)\nIMAGENET_STD_NP = np.array([0.229, 0.224, 0.225], dtype=np.float32)\nEXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}\nLABELS = {"real": 0, "fake": 1}\n\n\ndef setup_ddp():\n    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:\n        rank = int(os.environ["RANK"])\n        world_size = int(os.environ["WORLD_SIZE"])\n        local_rank = int(os.environ.get("LOCAL_RANK", 0))\n        torch.cuda.set_device(local_rank)\n        dist.init_process_group(backend="nccl", init_method="env://")\n        return True, rank, world_size, local_rank, torch.device(f"cuda:{local_rank}")\n    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")\n    return False, 0, 1, 0, dev\n\n\ndef cleanup_ddp(is_ddp: bool):\n    if is_ddp and dist.is_initialized():\n        dist.destroy_process_group()\n\n\ndef set_seed(seed: int):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef get_tmp_cache_dir(working: Path) -> Path:\n    if Path("/kaggle/working").exists() and Path("/tmp").exists():\n        d = Path("/tmp/ow_masks_cache")\n    else:\n        d = working / "_tmp_cache"\n    d.mkdir(parents=True, exist_ok=True)\n    return d\n\n\ndef discover_wild_root() -> Path:\n    roots = []\n    if Path("/kaggle/input").exists():\n        roots.append(Path("/kaggle/input"))\n    roots.extend([Path.cwd() / "openforensics-wild-dataset", Path.cwd() / "Dataset", Path.cwd()])\n    for root in roots:\n        if not root.exists():\n            continue\n        for dirpath, dirnames, _ in os.walk(root):\n            p = Path(dirpath)\n            if "ow_masks" in p.parts:\n                continue\n            kids = {d.lower() for d in dirnames}\n            if {"train", "validation", "test"} <= kids or {"train", "val", "test"} <= kids:\n                return p\n            if p.name.lower() == "dataset" and ("train" in kids or "test" in kids):\n                return p\n    raise FileNotFoundError("Attach ahmedgamal/openforensics-wild-dataset.")\n\n\ndef _split_dir(root: Path, name: str) -> Path:\n    aliases = {"train": ["Train", "train"], "val": ["Validation", "Val", "validation", "val"], "test": ["Test", "test"]}\n    for a in aliases[name]:\n        p = root / a\n        if p.is_dir():\n            return p\n    raise FileNotFoundError(f"No {name} split under {root}")\n\n\ndef _class_dir(split_dir: Path, name: str):\n    for a in (name, name.capitalize(), name.upper(), name.lower()):\n        p = split_dir / a\n        if p.is_dir():\n            return p\n    return None\n\n\ndef list_images(wild_root: Path, split_name: str) -> pd.DataFrame:\n    split_dir = _split_dir(wild_root, split_name)\n    rows = []\n    for cname, lab in LABELS.items():\n        cdir = _class_dir(split_dir, cname)\n        if cdir is None:\n            continue\n        for p in sorted(cdir.iterdir()):\n            if p.is_file() and p.suffix.lower() in EXTS:\n                rows.append({\n                    "path": str(p),\n                    "split": split_name,\n                    "label": lab,\n                    "class_name": cname,\n                    "file_name": p.name,\n                    "rel_key": f"{split_name}/{cname}/{p.name}",\n                })\n    return pd.DataFrame(rows)\n\n\ndef unpack_mask_outputs(input_root: Path, cache_dir: Path) -> list:\n    roots = []\n    if not input_root.exists():\n        return roots\n    for archive in sorted(input_root.rglob("_output_.zip")):\n        key = hashlib.sha256(str(archive).encode()).hexdigest()[:12]\n        target = cache_dir / f"unpacked_masks_{key}"\n        marker = target / ".done"\n        if marker.exists():\n            roots.append(target)\n            continue\n        t0 = time.time()\n        with zipfile.ZipFile(archive) as z:\n            members = [\n                m for m in z.infolist()\n                if not m.is_dir() and "ow_masks" in PurePosixPath(m.filename.replace("\\\\", "/")).parts\n            ]\n            if not members:\n                continue\n            print(f"[CACHE] Unpacking {len(members)} mask files to {target} ...", flush=True)\n            target.mkdir(parents=True, exist_ok=True)\n            for member in members:\n                rel = PurePosixPath(member.filename.replace("\\\\", "/"))\n                if rel.is_absolute() or ".." in rel.parts:\n                    continue\n                rel = Path(*rel.parts[rel.parts.index("ow_masks"):])\n                dst = target / rel\n                dst.parent.mkdir(parents=True, exist_ok=True)\n                with z.open(member) as src, dst.open("wb") as out:\n                    shutil.copyfileobj(src, out)\n            marker.write_text("ok", encoding="utf-8")\n            print(f"[CACHE] Unpacked masks to /tmp in {time.time() - t0:.1f}s", flush=True)\n            roots.append(target)\n    return roots\n\n\ndef load_mask_table(tmp_cache: Path, working: Path) -> pd.DataFrame:\n    frames = []\n    roots = [p for p in (Path("/kaggle/input"), tmp_cache, Path.cwd()) if p.exists()]\n    roots.extend(unpack_mask_outputs(Path("/kaggle/input"), tmp_cache))\n    seen = set()\n    for root in roots:\n        for dirpath, _, filenames in os.walk(root):\n            if "index.csv" not in filenames:\n                continue\n            path = Path(dirpath) / "index.csv"\n            if str(path) in seen:\n                continue\n            seen.add(str(path))\n            try:\n                df = pd.read_csv(path, dtype=str)\n            except Exception:\n                continue\n            if not {"image_id", "rel_key", "mask_file", "status", "crop_x0"} <= set(df.columns):\n                continue\n            df["_root"] = str(path.parent)\n            frames.append(df)\n    if not frames:\n        raise FileNotFoundError("No ow_masks/index.csv found. Attach clarenceshanley/veritas-ow-00-generate-masks.")\n    all_df = pd.concat(frames, ignore_index=True)\n    kept, used = [], set()\n    for rec in all_df.to_dict("records"):\n        iid = str(rec["image_id"])\n        if iid in used or str(rec.get("status")) != "ok":\n            continue\n        png = Path(str(rec["_root"])) / str(rec["mask_file"])\n        if not png.is_file():\n            continue\n        rec["mask_path"] = str(png)\n        kept.append(rec)\n        used.add(iid)\n    return pd.DataFrame(kept).drop_duplicates("rel_key", keep="last")\n\n\ndef build_index(cfg: dict, tmp_cache: Path, working: Path, rank: int) -> dict:\n    cache_path = tmp_cache / "index_v41_140k.json"\n    if rank == 0:\n        wild_root = discover_wild_root()\n        masks = load_mask_table(tmp_cache, working)\n        index = {}\n        limits = {"train": cfg["train_per_class"], "val": cfg["val_per_class"], "test": cfg["test_per_class"]}\n        for split in ("train", "val", "test"):\n            imgs = list_images(wild_root, split)\n            merged = imgs.merge(\n                masks[["rel_key", "mask_path", "crop_x0", "crop_y0", "crop_x1", "crop_y1", "positive_frac"]],\n                on="rel_key",\n                how="left",\n            )\n            ready = merged.loc[merged["mask_path"].notna()].copy()\n            lim = limits[split]\n            if lim is not None:\n                parts = []\n                rng = np.random.RandomState(cfg["seed"] + {"train": 11, "val": 22, "test": 33}[split])\n                for _, g in ready.groupby("label"):\n                    n = min(int(lim), len(g))\n                    parts.append(g.sample(n=n, random_state=rng) if n < len(g) else g)\n                ready = pd.concat(parts)\n            ready = ready.reset_index(drop=True)\n            index[split] = ready.to_dict("records")\n            print(\n                f"[INDEX] {split}: {len(ready)} images "\n                f"(real={(ready.label==0).sum()}, fake={(ready.label==1).sum()})",\n                flush=True,\n            )\n        cache_path.write_text(json.dumps(index), encoding="utf-8")\n    if dist.is_initialized():\n        dist.barrier()\n    loaded = json.loads(cache_path.read_text(encoding="utf-8"))\n    return {k: pd.DataFrame(v) for k, v in loaded.items()}\n\n\ndef letterbox_rgb(image_rgb: np.ndarray, size: int) -> tuple[np.ndarray, np.ndarray]:\n    h, w = image_rgb.shape[:2]\n    scale = min(size / max(h, 1), size / max(w, 1))\n    nw, nh = max(int(round(w * scale)), 1), max(int(round(h * scale)), 1)\n    resized = cv2.resize(image_rgb, (nw, nh), interpolation=cv2.INTER_LINEAR)\n    canvas = np.zeros((size, size, 3), dtype=np.uint8)\n    valid = np.zeros((size, size), dtype=np.float32)\n    left, top = (size - nw) // 2, (size - nh) // 2\n    canvas[top : top + nh, left : left + nw] = resized\n    valid[top : top + nh, left : left + nw] = 1.0\n    return canvas, valid\n\n\ndef apply_forensic_augmentations(crop: np.ndarray, rng: random.Random, forensic_aug: bool) -> np.ndarray:\n    if rng.random() < 0.45:\n        alpha = 1.0 + rng.uniform(-0.15, 0.15)\n        beta = rng.uniform(-12.0, 12.0)\n        crop = np.clip(crop.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)\n    if not forensic_aug:\n        return crop\n    if rng.random() < 0.25:\n        h, w = crop.shape[:2]\n        sc = rng.uniform(0.65, 0.95)\n        small = cv2.resize(crop, (max(16, int(w * sc)), max(16, int(h * sc))), interpolation=cv2.INTER_AREA)\n        crop = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)\n    if rng.random() < 0.20:\n        k = rng.choice([3, 5])\n        crop = cv2.GaussianBlur(crop, (k, k), 0)\n    if rng.random() < 0.45:\n        q = rng.randint(45, 95)\n        ok, enc = cv2.imencode(".jpg", cv2.cvtColor(crop, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), q])\n        if ok:\n            dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)\n            if dec is not None:\n                crop = cv2.cvtColor(dec, cv2.COLOR_BGR2RGB)\n    return crop\n\n\ndef apply_corruption(crop: np.ndarray, condition: str) -> np.ndarray:\n    if condition == "clean":\n        return crop\n    if condition == "jpeg_q50":\n        ok, enc = cv2.imencode(".jpg", cv2.cvtColor(crop, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), 50])\n        if ok:\n            dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)\n            if dec is not None:\n                return cv2.cvtColor(dec, cv2.COLOR_BGR2RGB)\n        return crop\n    if condition == "jpeg_q30":\n        ok, enc = cv2.imencode(".jpg", cv2.cvtColor(crop, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), 30])\n        if ok:\n            dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)\n            if dec is not None:\n                return cv2.cvtColor(dec, cv2.COLOR_BGR2RGB)\n        return crop\n    if condition == "blur_sigma2":\n        return cv2.GaussianBlur(crop, (0, 0), sigmaX=2.0, sigmaY=2.0)\n    if condition == "resize_0.5x":\n        h, w = crop.shape[:2]\n        small = cv2.resize(crop, (max(16, w // 2), max(16, h // 2)), interpolation=cv2.INTER_AREA)\n        return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)\n    raise ValueError(f"Unknown condition: {condition}")\n\n\nclass OpenForensicsMaskDataset(Dataset):\n    def __init__(self, df: pd.DataFrame, size: int, augment: bool, forensic_aug: bool, seed: int, condition: str = "clean"):\n        self.df = df.reset_index(drop=True)\n        self.size = int(size)\n        self.augment = bool(augment)\n        self.forensic_aug = bool(forensic_aug)\n        self.seed = int(seed)\n        self.condition = str(condition)\n        self.epoch = 0\n\n    def __len__(self):\n        return len(self.df)\n\n    def __getitem__(self, idx: int):\n        row = self.df.iloc[idx]\n        rng = random.Random(self.seed + self.epoch * 1_000_003 + int(idx))\n        bgr = cv2.imread(row.path, cv2.IMREAD_COLOR)\n        if bgr is None:\n            raise FileNotFoundError(f"Unreadable image: {row.path}")\n        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)\n        h, w = rgb.shape[:2]\n        x0, y0 = max(0, int(float(row.crop_x0))), max(0, int(float(row.crop_y0)))\n        x1, y1 = min(w, max(x0 + 1, int(float(row.crop_x1)))), min(h, max(y0 + 1, int(float(row.crop_y1))))\n        crop = rgb[y0:y1, x0:x1]\n\n        mask = cv2.imread(str(row.mask_path), cv2.IMREAD_GRAYSCALE)\n        if mask is None:\n            raise FileNotFoundError(f"Unreadable mask: {row.mask_path}")\n        if mask.shape != (self.size, self.size):\n            mask = cv2.resize(mask, (self.size, self.size), interpolation=cv2.INTER_NEAREST)\n\n        if self.augment:\n            crop = apply_forensic_augmentations(crop, rng, self.forensic_aug)\n        elif self.condition != "clean":\n            crop = apply_corruption(crop, self.condition)\n\n        canvas, valid = letterbox_rgb(crop, self.size)\n        if self.augment and rng.random() < 0.5:\n            canvas = np.ascontiguousarray(canvas[:, ::-1])\n            mask = np.ascontiguousarray(mask[:, ::-1])\n            valid = np.ascontiguousarray(valid[:, ::-1])\n\n        x = (canvas.astype(np.float32) / 255.0 - IMAGENET_MEAN_NP) / IMAGENET_STD_NP\n        x = np.transpose(x, (2, 0, 1))\n        target = (mask.astype(np.float32) > 127.0).astype(np.float32)\n        return {\n            "image": torch.from_numpy(x.copy()),\n            "mask": torch.from_numpy(target[None].copy()),\n            "valid_mask": torch.from_numpy(valid[None].copy()),\n            "label": torch.tensor([float(row.label)], dtype=torch.float32),\n            "rel_key": str(row.rel_key),\n            "file_name": str(row.file_name),\n        }\n\n\nclass VERITASLossV41(nn.Module):\n    def __init__(self, lambda_cls=1.0, lambda_rgb=0.50, lambda_aux=0.50, lambda_seg=0.50, dice_weight=1.0, variant="veritas_v4_1"):\n        super().__init__()\n        self.lambda_cls = float(lambda_cls)\n        self.lambda_rgb = float(lambda_rgb)\n        self.lambda_aux = float(lambda_aux)\n        self.lambda_seg = float(lambda_seg)\n        self.dice_weight = float(dice_weight)\n        self.variant = str(variant)\n\n    def forward(self, out: dict, labels: torch.Tensor, masks: torch.Tensor, valid_mask: torch.Tensor) -> dict:\n        labels_f = labels.float()\n        cls_loss = F.binary_cross_entropy_with_logits(out["classification_logits"].float(), labels_f)\n        rgb_loss = cls_loss.new_zeros(())\n        aux_loss = cls_loss.new_zeros(())\n        seg_loss = cls_loss.new_zeros(())\n\n        if self.variant != "baseline":\n            if "rgb_logits" in out:\n                rgb_loss = F.binary_cross_entropy_with_logits(out["rgb_logits"].float(), labels_f)\n            aux_loss = F.binary_cross_entropy_with_logits(out["aux_forensic_logits"].float(), labels_f)\n            seg_logits = out["segmentation_logits"].float()\n            targets = masks.float()\n            real = labels_f[:, 0] == 0\n            targets = torch.where(real[:, None, None, None], torch.zeros_like(targets), targets)\n            valid = valid_mask.float()\n            dims = (1, 2, 3)\n            px = valid.sum(dims).clamp_min(1.0)\n            nonempty = (targets * valid).sum(dims) > 0\n            trusted = real | nonempty\n            positive = (~real) & nonempty\n\n            per_bce = (F.binary_cross_entropy_with_logits(seg_logits, targets, reduction="none") * valid).sum(dims) / px\n            probs = seg_logits.sigmoid() * valid\n            fg = targets * valid\n            per_dice = 1.0 - (2.0 * (probs * fg).sum(dims) + 1e-6) / (probs.sum(dims) + fg.sum(dims) + 1e-6)\n            per_img = torch.where(positive, (per_bce + self.dice_weight * per_dice) / (1.0 + self.dice_weight), per_bce)\n            tw = trusted.float()\n            seg_loss = (per_img * tw).sum() / tw.sum().clamp_min(1.0)\n\n        total = (\n            self.lambda_cls * cls_loss\n            + self.lambda_rgb * rgb_loss\n            + self.lambda_aux * aux_loss\n            + self.lambda_seg * seg_loss\n        )\n        return {\n            "total": total,\n            "cls": cls_loss.detach(),\n            "rgb": rgb_loss.detach(),\n            "aux": aux_loss.detach(),\n            "seg": seg_loss.detach(),\n        }\n\n\ndef freeze_backbone_bn(model: nn.Module):\n    enc = model.module.encoder if isinstance(model, DDP) else model.encoder\n    for m in enc.modules():\n        if isinstance(m, nn.modules.batchnorm._BatchNorm):\n            m.eval()\n\n\ndef build_optimizer(model: nn.Module, lr: float, weight_decay: float, backbone_mult: float):\n    raw = model.module if isinstance(model, DDP) else model\n    bb = [p for p in raw.encoder.parameters() if p.requires_grad]\n    bb_ids = {id(p) for p in bb}\n    heads = [p for p in raw.parameters() if p.requires_grad and id(p) not in bb_ids]\n    groups = []\n    if bb:\n        groups.append({"params": bb, "lr": lr * backbone_mult})\n    if heads:\n        groups.append({"params": heads, "lr": lr})\n    return torch.optim.AdamW(groups, lr=lr, weight_decay=weight_decay)\n\n\n@torch.no_grad()\ndef evaluate(model: nn.Module, loader: DataLoader, device: torch.device, is_ddp: bool, variant: str, use_amp: bool):\n    model.eval()\n    is_veritas = variant != "baseline"\n    has_rgb_head = variant == "veritas_v4_1"\n    local_recs = []\n    for batch in loader:\n        x = batch["image"].to(device, non_blocking=True).to(memory_format=torch.channels_last)\n        vm = batch["valid_mask"].to(device, non_blocking=True)\n        masks = batch["mask"].to(device, non_blocking=True)\n        labels = batch["label"].to(device, non_blocking=True)\n        with torch.amp.autocast("cuda", enabled=(use_amp and device.type == "cuda")):\n            out = model(x, vm)\n        probs = out["fake_probability"].float().squeeze(1).cpu().numpy()\n        y = labels.squeeze(1).cpu().numpy().astype(int)\n        rgb_p = out["rgb_logits"].float().sigmoid().squeeze(1).cpu().numpy() if has_rgb_head else None\n        aux_p = out["aux_forensic_logits"].float().sigmoid().squeeze(1).cpu().numpy() if is_veritas else None\n        gates = out["stream_gates"].float().cpu().numpy() if is_veritas else None\n\n        real_pos_frac = [None] * len(y)\n        fake_dice = [None] * len(y)\n        fake_iou = [None] * len(y)\n        if is_veritas:\n            pred_m = (out["segmentation_prob"].float() >= 0.5).float() * vm\n            gt_m = masks.float() * vm\n            v_sum = vm.sum(dim=(1, 2, 3)).clamp_min(1.0)\n            p_sum = pred_m.sum(dim=(1, 2, 3))\n            g_sum = gt_m.sum(dim=(1, 2, 3))\n            inter = (pred_m * gt_m).sum(dim=(1, 2, 3))\n            union = (pred_m + gt_m - pred_m * gt_m).sum(dim=(1, 2, 3))\n            rpf = (p_sum / v_sum).cpu().numpy()\n            dice_arr = ((2.0 * inter + 1e-6) / (p_sum + g_sum + 1e-6)).cpu().numpy()\n            iou_arr = ((inter + 1e-6) / (union + 1e-6)).cpu().numpy()\n            for i in range(len(y)):\n                if y[i] == 0:\n                    real_pos_frac[i] = float(rpf[i])\n                else:\n                    fake_dice[i] = float(dice_arr[i])\n                    fake_iou[i] = float(iou_arr[i])\n\n        for i in range(len(y)):\n            rec = {\n                "rel_key": batch["rel_key"][i],\n                "file_name": batch["file_name"][i],\n                "ground_truth_label": int(y[i]),\n                "fake_probability": float(probs[i]),\n                "predicted_label": int(probs[i] >= 0.5),\n            }\n            if is_veritas:\n                if has_rgb_head:\n                    rec["rgb_probability"] = float(rgb_p[i])\n                rec["aux_forensic_probability"] = float(aux_p[i])\n                rec["real_positive_fraction"] = real_pos_frac[i]\n                rec["dice_teacher_fake"] = fake_dice[i]\n                rec["iou_teacher_fake"] = fake_iou[i]\n                rec["gate_rgb"] = float(gates[i, 0])\n                rec["gate_srm"] = float(gates[i, 1])\n            local_recs.append(rec)\n\n    if is_ddp:\n        gathered = [None for _ in range(dist.get_world_size())]\n        dist.all_gather_object(gathered, local_recs)\n        merged = [r for sub in gathered for r in sub]\n    else:\n        merged = local_recs\n\n    dedup = {r["rel_key"]: r for r in merged}\n    recs = [dedup[k] for k in sorted(dedup.keys())]\n\n    y_true = np.array([r["ground_truth_label"] for r in recs], dtype=int)\n    p = np.array([r["fake_probability"] for r in recs], dtype=float)\n    y_pred = (p >= 0.5).astype(int)\n    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()\n    metrics = {\n        "n": int(len(recs)),\n        "accuracy": float(accuracy_score(y_true, y_pred)),\n        "f1": float(f1_score(y_true, y_pred, zero_division=0)),\n        "precision": float(precision_score(y_true, y_pred, zero_division=0)),\n        "recall": float(recall_score(y_true, y_pred, zero_division=0)),\n        "roc_auc": float(roc_auc_score(y_true, p)),\n        "tp": int(tp),\n        "tn": int(tn),\n        "fp": int(fp),\n        "fn": int(fn),\n    }\n    if is_veritas:\n        aux_p = np.array([r["aux_forensic_probability"] for r in recs], dtype=float)\n        metrics["aux_forensic_accuracy"] = float(accuracy_score(y_true, (aux_p >= 0.5).astype(int)))\n        metrics["aux_forensic_roc_auc"] = float(roc_auc_score(y_true, aux_p))\n        if has_rgb_head:\n            rgb_p = np.array([r["rgb_probability"] for r in recs], dtype=float)\n            metrics["rgb_accuracy"] = float(accuracy_score(y_true, (rgb_p >= 0.5).astype(int)))\n            metrics["rgb_roc_auc"] = float(roc_auc_score(y_true, rgb_p))\n            eps = 1e-6\n            lr = np.log(np.clip(rgb_p, eps, 1 - eps) / (1 - np.clip(rgb_p, eps, 1 - eps)))\n            la = np.log(np.clip(aux_p, eps, 1 - eps) / (1 - np.clip(aux_p, eps, 1 - eps)))\n            metrics["logit_corr_rgb_aux"] = float(np.corrcoef(lr, la)[0, 1]) if len(lr) > 2 else float("nan")\n        rpf_vals = [r["real_positive_fraction"] for r in recs if r["real_positive_fraction"] is not None]\n        d_vals = [r["dice_teacher_fake"] for r in recs if r["dice_teacher_fake"] is not None]\n        i_vals = [r["iou_teacher_fake"] for r in recs if r["iou_teacher_fake"] is not None]\n        metrics["real_mean_positive_fraction"] = float(np.mean(rpf_vals)) if rpf_vals else float("nan")\n        metrics["real_mask_nonempty_rate"] = float(np.mean([v > 0 for v in rpf_vals])) if rpf_vals else float("nan")\n        metrics["dice_teacher_fake_only"] = float(np.mean(d_vals)) if d_vals else float("nan")\n        metrics["miou_teacher_fake_only"] = float(np.mean(i_vals)) if i_vals else float("nan")\n        metrics["mean_gate_rgb"] = float(np.mean([r["gate_rgb"] for r in recs]))\n        metrics["mean_gate_srm"] = float(np.mean([r["gate_srm"] for r in recs]))\n    return metrics, recs\n\n\n@torch.no_grad()\ndef benchmark_inference(model: nn.Module, device: torch.device, batch_size: int = 4, size: int = 600):\n    raw = model.module if isinstance(model, DDP) else model\n    raw.eval()\n    x = torch.randn(batch_size, 3, size, size, device=device)\n    vm = torch.ones(batch_size, 1, size, size, device=device)\n    if device.type == "cuda":\n        torch.cuda.reset_peak_memory_stats(device)\n        torch.cuda.synchronize(device)\n    with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):\n        _ = raw(x, vm)\n    if device.type == "cuda":\n        torch.cuda.synchronize(device)\n    t0 = time.perf_counter()\n    repeats = 5\n    for _ in range(repeats):\n        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):\n            _ = raw(x, vm)\n    if device.type == "cuda":\n        torch.cuda.synchronize(device)\n    batch_ms = ((time.perf_counter() - t0) / repeats) * 1000.0\n    peak_bytes = int(torch.cuda.max_memory_allocated(device)) if device.type == "cuda" else 0\n    return {\n        f"per_image_ms_bs{batch_size}": batch_ms / batch_size,\n        f"batch_latency_ms_bs{batch_size}": batch_ms,\n        f"peak_vram_mib_bs{batch_size}": peak_bytes / (1024 * 1024),\n    }\n\n\ndef train_one_stage(\n    model: nn.Module,\n    loader: DataLoader,\n    sampler,\n    criterion: VERITASLossV41,\n    optimizer: torch.optim.Optimizer,\n    scheduler,\n    scaler,\n    device: torch.device,\n    epoch: int,\n    grad_accum: int,\n    use_amp: bool,\n    rank: int,\n    stage_name: str,\n):\n    model.train()\n    freeze_backbone_bn(model)\n    loader.dataset.epoch = epoch\n    if sampler is not None:\n        sampler.set_epoch(epoch)\n    optimizer.zero_grad(set_to_none=True)\n    t0 = time.time()\n    n_steps = len(loader)\n    tot_loss = tot_cls = tot_rgb = tot_aux = tot_seg = 0.0\n    log_every = max(1, n_steps // 10)\n    for step, batch in enumerate(loader, start=1):\n        x = batch["image"].to(device, non_blocking=True).to(memory_format=torch.channels_last)\n        vm = batch["valid_mask"].to(device, non_blocking=True)\n        masks = batch["mask"].to(device, non_blocking=True)\n        labels = batch["label"].to(device, non_blocking=True)\n        with torch.amp.autocast("cuda", enabled=(use_amp and device.type == "cuda")):\n            out = model(x, vm)\n            losses = criterion(out, labels, masks, vm)\n            scaled_loss = losses["total"] / grad_accum\n        scaler.scale(scaled_loss).backward()\n        if step % grad_accum == 0 or step == n_steps:\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)\n            scaler.step(optimizer)\n            scaler.update()\n            optimizer.zero_grad(set_to_none=True)\n            scheduler.step()\n        tot_loss += float(losses["total"].detach())\n        tot_cls += float(losses["cls"])\n        tot_rgb += float(losses["rgb"])\n        tot_aux += float(losses["aux"])\n        tot_seg += float(losses["seg"])\n        if rank == 0 and (step % log_every == 0 or step == n_steps):\n            dt = time.time() - t0\n            eta = (dt / step) * (n_steps - step)\n            print(\n                f"  [{stage_name} ep {epoch}] step {step}/{n_steps} "\n                f"loss={tot_loss/step:.4f} cls={tot_cls/step:.4f} rgb={tot_rgb/step:.4f} "\n                f"aux={tot_aux/step:.4f} seg={tot_seg/step:.4f} ({dt:.0f}s, eta {eta/60:.1f}m)",\n                flush=True,\n            )\n    return time.time() - t0\n\n\ndef run_single_variant(variant: str, cfg: dict, index: dict, working: Path, is_ddp: bool, rank: int, world_size: int, local_rank: int, device: torch.device):\n    set_seed(cfg["seed"])\n    is_veritas = variant != "baseline"\n    t_start = time.time()\n    if rank == 0:\n        print(f"\\n==================== RUNNING FULL-140K {variant.upper()} ====================", flush=True)\n\n    train_ds = OpenForensicsMaskDataset(index["train"], cfg["image_size"], augment=True, forensic_aug=cfg["use_forensic_aug"], seed=cfg["seed"])\n    val_ds = OpenForensicsMaskDataset(index["val"], cfg["image_size"], augment=False, forensic_aug=False, seed=cfg["seed"])\n    test_ds = OpenForensicsMaskDataset(index["test"], cfg["image_size"], augment=False, forensic_aug=False, seed=cfg["seed"])\n\n    tr_samp = DistributedSampler(train_ds, num_replicas=world_size, rank=rank, shuffle=True, seed=cfg["seed"], drop_last=True) if is_ddp else None\n    va_samp = DistributedSampler(val_ds, num_replicas=world_size, rank=rank, shuffle=False, drop_last=False) if is_ddp else None\n    te_samp = DistributedSampler(test_ds, num_replicas=world_size, rank=rank, shuffle=False, drop_last=False) if is_ddp else None\n\n    nw = cfg["num_workers"]\n    eval_bs = cfg.get("eval_batch_size_per_gpu", cfg["batch_size_per_gpu"])\n    kw = dict(num_workers=nw, pin_memory=(device.type == "cuda"), persistent_workers=(nw > 0))\n    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size_per_gpu"], sampler=tr_samp, shuffle=(tr_samp is None), drop_last=True, **kw)\n    val_loader = DataLoader(val_ds, batch_size=eval_bs, sampler=va_samp, shuffle=False, **kw)\n    test_loader = DataLoader(test_ds, batch_size=eval_bs, sampler=te_samp, shuffle=False, **kw)\n\n    grad_accum = max(1, cfg["target_effective_batch"] // (cfg["batch_size_per_gpu"] * world_size))\n\n    if is_ddp:\n        if rank == 0:\n            _ = efficientnet_b7(weights=EfficientNet_B7_Weights.IMAGENET1K_V1)\n        dist.barrier()\n    raw_model = VERITASV4(variant=variant, pretrained=True).to(device).to(memory_format=torch.channels_last)\n    criterion = VERITASLossV41(\n        lambda_cls=cfg["lambda_cls"],\n        lambda_rgb=cfg.get("lambda_rgb", 0.50),\n        lambda_aux=cfg["lambda_aux"],\n        lambda_seg=cfg["lambda_seg"],\n        dice_weight=cfg["dice_weight"],\n        variant=variant,\n    )\n    scaler = torch.amp.GradScaler("cuda", enabled=(cfg["use_amp"] and device.type == "cuda")) if hasattr(torch.amp, "GradScaler") else torch.cuda.amp.GradScaler(enabled=(cfg["use_amp"] and device.type == "cuda"))\n\n    best_val_acc = -1.0\n    best_path = working / "checkpoints" / f"{variant}_140k_best.pt"\n    best_path.parent.mkdir(parents=True, exist_ok=True)\n    pred_dir = working / "predictions"\n    pred_dir.mkdir(parents=True, exist_ok=True)\n    history = []\n\n    def log_val(stage_label: str, ep_num: int, sec: float, val_m: dict, val_recs: list):\n        nonlocal best_val_acc\n        extra = ""\n        if is_veritas:\n            extra = (\n                f" rgb={val_m.get(\'rgb_accuracy\', 0):.4f} aux={val_m[\'aux_forensic_accuracy\']:.4f} "\n                f"r={val_m.get(\'logit_corr_rgb_aux\', 0):.3f} alpha={val_m[\'mean_gate_rgb\']:.3f} "\n                f"dice={val_m[\'dice_teacher_fake_only\']:.4f}"\n            )\n        print(\n            f"  [{stage_label} ep {ep_num}] VAL acc={val_m[\'accuracy\']:.4f} f1={val_m[\'f1\']:.4f} "\n            f"auc={val_m[\'roc_auc\']:.4f}{extra} ({sec/60:.1f}m)",\n            flush=True,\n        )\n        history.append({"stage": stage_label, "epoch": ep_num, "seconds": sec, **val_m})\n        if val_m["accuracy"] > best_val_acc:\n            best_val_acc = val_m["accuracy"]\n            torch.save(raw_model.state_dict(), best_path)\n            pd.DataFrame(val_recs).to_csv(pred_dir / f"{variant}_140k_val_predictions.csv", index=False)\n            (working / f"{variant}_140k_progress.json").write_text(\n                json.dumps({"best_val_acc": best_val_acc, "history": history}, indent=2), encoding="utf-8"\n            )\n\n    # Stage 1: Frozen-backbone head warmup\n    if cfg["warmup_epochs"] > 0:\n        raw_model.freeze_backbone(True)\n        model = DDP(raw_model, device_ids=[local_rank], find_unused_parameters=False) if is_ddp else raw_model\n        opt = build_optimizer(model, lr=cfg["warmup_lr"], weight_decay=cfg["weight_decay"], backbone_mult=1.0)\n        updates = max(1, math.ceil(len(train_loader) / grad_accum) * cfg["warmup_epochs"])\n        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=updates, eta_min=cfg["warmup_lr"] * 0.1)\n        for ep in range(1, cfg["warmup_epochs"] + 1):\n            sec = train_one_stage(model, train_loader, tr_samp, criterion, opt, sched, scaler, device, ep, grad_accum, cfg["use_amp"], rank, "Stage1-Warmup")\n            val_m, val_recs = evaluate(raw_model, val_loader, device, is_ddp, variant, cfg["use_amp"])\n            if rank == 0:\n                log_val("warmup", ep, sec, val_m, val_recs)\n        del model, opt, sched\n        gc.collect()\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n    # Stage 2: Full end-to-end fine-tuning\n    if cfg["finetune_epochs"] > 0:\n        raw_model.freeze_backbone(False)\n        model = DDP(raw_model, device_ids=[local_rank], find_unused_parameters=False) if is_ddp else raw_model\n        opt = build_optimizer(model, lr=cfg["finetune_lr"], weight_decay=cfg["weight_decay"], backbone_mult=cfg["backbone_lr_multiplier"])\n        updates = max(1, math.ceil(len(train_loader) / grad_accum) * cfg["finetune_epochs"])\n        warmup_u = max(1, int(round(updates * 0.05)))\n        def lr_fn(u):\n            if u < warmup_u:\n                return (u + 1) / warmup_u\n            prog = min(1.0, (u - warmup_u) / max(1, updates - warmup_u))\n            return 0.5 * (1.0 + math.cos(math.pi * prog))\n        sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)\n        for ep in range(1, cfg["finetune_epochs"] + 1):\n            sec = train_one_stage(model, train_loader, tr_samp, criterion, opt, sched, scaler, device, cfg["warmup_epochs"] + ep, grad_accum, cfg["use_amp"], rank, "Stage2-Finetune")\n            val_m, val_recs = evaluate(raw_model, val_loader, device, is_ddp, variant, cfg["use_amp"])\n            if rank == 0:\n                log_val("finetune", ep, sec, val_m, val_recs)\n        del model, opt, sched\n        gc.collect()\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n    if is_ddp:\n        dist.barrier()\n\n    # Clean evaluation on ALL 10,905 OpenForensics Test images\n    raw_model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))\n    test_m, test_recs = evaluate(raw_model, test_loader, device, is_ddp, variant, cfg["use_amp"])\n    train_eval_sec = time.time() - t_start\n\n    if rank == 0:\n        bench1 = benchmark_inference(raw_model, device, batch_size=1, size=cfg["image_size"])\n        bench4 = benchmark_inference(raw_model, device, batch_size=4, size=cfg["image_size"])\n        test_m.update(bench1)\n        test_m.update(bench4)\n        test_m["total_train_eval_seconds"] = train_eval_sec\n        test_m["variant"] = variant\n        if variant == "veritas_v4_1":\n            test_m["learned_alpha"] = float((0.35 + 0.30 * torch.sigmoid(raw_model.fusion_head.raw_alpha)).item())\n            test_m["learned_res_scale"] = float((0.25 * torch.tanh(raw_model.fusion_head.raw_res_scale)).item())\n        pd.DataFrame(test_recs).to_csv(pred_dir / f"{variant}_140k_test_predictions.csv", index=False)\n        (working / f"{variant}_140k_summary.json").write_text(\n            json.dumps({"metrics": test_m, "history": history, "config": cfg}, indent=2), encoding="utf-8"\n        )\n        print(f"=== FINAL CLEAN TEST ({variant}) ===\\n" + json.dumps(test_m, indent=2), flush=True)\n\n    del train_loader, val_loader, test_loader\n    gc.collect()\n\n    # Built-in 5-Condition Robustness Stress Test on ALL 10,905 Test images (guarded by wall-clock budget)\n    robustness = {"clean": test_m}\n    budget_sec = float(cfg.get("robustness_time_budget_sec", 37800))\n    eval_kw = dict(num_workers=nw, pin_memory=(device.type == "cuda"), persistent_workers=False)\n    for cond in cfg.get("robustness_conditions", ["jpeg_q50", "jpeg_q30", "blur_sigma2", "resize_0.5x"]):\n        elapsed = time.time() - t_start\n        # Broadcast budget decision from rank 0 so all DDP ranks stay synchronized\n        go = torch.tensor([1 if elapsed < budget_sec else 0], device=device)\n        if is_ddp:\n            dist.broadcast(go, src=0)\n        if int(go.item()) == 0:\n            if rank == 0:\n                print(f"[ROBUSTNESS] Skipping {cond}: elapsed {elapsed/3600:.2f}h >= budget {budget_sec/3600:.2f}h", flush=True)\n            break\n        tc = time.time()\n        cond_ds = OpenForensicsMaskDataset(index["test"], cfg["image_size"], augment=False, forensic_aug=False, seed=cfg["seed"], condition=cond)\n        cond_samp = DistributedSampler(cond_ds, num_replicas=world_size, rank=rank, shuffle=False, drop_last=False) if is_ddp else None\n        cond_loader = DataLoader(cond_ds, batch_size=eval_bs, sampler=cond_samp, shuffle=False, **eval_kw)\n        cm, crecs = evaluate(raw_model, cond_loader, device, is_ddp, variant, cfg["use_amp"])\n        if rank == 0:\n            cm["eval_seconds"] = time.time() - tc\n            robustness[cond] = cm\n            pd.DataFrame(crecs).to_csv(pred_dir / f"{variant}_140k_robust_{cond}.csv", index=False)\n            (working / f"{variant}_140k_robustness_summary.json").write_text(json.dumps(robustness, indent=2), encoding="utf-8")\n            print(\n                f"  [ROBUSTNESS {cond:12s}] Acc={cm[\'accuracy\']*100:.2f}% F1={cm[\'f1\']:.4f} "\n                f"AUC={cm[\'roc_auc\']:.4f} ({cm[\'eval_seconds\']:.0f}s)",\n                flush=True,\n            )\n\n    del raw_model\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", required=True)\n    args = parser.parse_args()\n    cfg = json.loads(Path(args.config).read_text(encoding="utf-8"))\n    working = Path(cfg["working_dir"])\n    working.mkdir(parents=True, exist_ok=True)\n    tmp_cache = get_tmp_cache_dir(working)\n\n    is_ddp, rank, world_size, local_rank, device = setup_ddp()\n    try:\n        if rank == 0:\n            print(\n                f"DDP={is_ddp} world_size={world_size} device={device} "\n                f"gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'cpu\'} "\n                f"tmp_cache={tmp_cache}",\n                flush=True,\n            )\n        index = build_index(cfg, tmp_cache, working, rank)\n        run_single_variant(cfg["variant"], cfg, index, working, is_ddp, rank, world_size, local_rank, device)\n    finally:\n        cleanup_ddp(is_ddp)\n\n\nif __name__ == "__main__":\n    main()\n'
script_path = WORKING / 'train_v41_140k.py'
script_path.write_text(SCRIPT_CODE, encoding='utf-8')
print('Wrote training script:', script_path, 'bytes:', len(SCRIPT_CODE))


In [ ]:
import subprocess
import sys

n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if n_gpus >= 2:
    cmd = ["torchrun", "--standalone", f"--nproc_per_node={n_gpus}", str(script_path), "--config", str(cfg_path)]
else:
    cmd = [sys.executable, str(script_path), "--config", str(cfg_path)]

print("Launching:", " ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"Training script failed with exit code {rc}")
print("Full-140k training, clean test, and robustness evaluation finished cleanly!")


In [ ]:
import json
import pandas as pd

summary = json.loads((WORKING / "veritas_v4_1_140k_summary.json").read_text())
print("=== CLEAN TEST SUMMARY (veritas_v4_1) ===")
print(json.dumps(summary["metrics"], indent=2))

rob_path = WORKING / "veritas_v4_1_140k_robustness_summary.json"
if rob_path.exists():
    rob = json.loads(rob_path.read_text())
    rows = []
    for cond, m in rob.items():
        rows.append({
            "condition": cond,
            "acc_pct": round(m["accuracy"] * 100, 2),
            "f1": round(m["f1"], 4),
            "prec": round(m["precision"], 4),
            "rec": round(m["recall"], 4),
            "auc": round(m["roc_auc"], 4),
            "fp": m["fp"],
            "fn": m["fn"],
        })
    print("\n=== 5-CONDITION ROBUSTNESS TABLE (10,905 TEST IMAGES) ===")
    print(pd.DataFrame(rows).to_string(index=False))
